# Food Data Analysis

In [1]:
#Importing essential libraries
from openpyxl import load_workbook
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import csv

#Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


In [2]:
food_data = pd.read_csv("food.csv")
food_data.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
3,319877,sub_sample_food,Hummus,16.0,2019-04-01
4,319878,sub_sample_food,Hummus,16.0,2019-04-01


In [3]:
food_matrix_5d = pd.read_csv("food_matrix_5d.csv")
food_matrix_5d.head()

,food_description,"Fiber, total dietary","Fatty acids, total polyunsaturated","Magnesium, Mg",Vitamin_D_Total_UG,"Zinc, Zn"
0,"Alaska Pollock, raw",0.000,0.4577,22.820,0.000000,0.431300
1,"Almond butter, creamy",9.718,12.6100,267.800,0.000000,3.178000
2,"Almond milk, unsweetened",0.000,0.2763,7.273,1.147117,0.140070
3,"Anchovies, canned in olive oil",0.000,0.0000,227.600,0.000000,2.539000
4,"Apple juice, with added vitamin C",0.000,0.0000,4.861,0.000000,0.002125


In [4]:
#Target nutrients matching to nutrients names in the USDA Nutrition/Food datasets

#Loading the nutrition datasets
df_food = pd.read_csv('food.csv')
df_food_nutrient = pd.read_csv('food_nutrient.csv')
df_nutrient = pd.read_csv('nutrient.csv')

#Seperating the target nutrients using the USDA nutrient ids
usda_ids = [291, 646, 304]
df_selected_nutrients = df_nutrient[df_nutrient['nutrient_nbr'].isin(usda_ids)]

#Relational Merge Pipeline
#Linking the filtered nutrients to the bridge table
merge_part1 = pd.merge(df_food_nutrient, df_selected_nutrients, left_on='nutrient_id', right_on='id', how='inner')

#Linking the result to food name table
df_joined = pd.merge(merge_part1, df_food, on='fdc_id', how='inner')

#Converting the table from a vertical format to horizontal format
#Each row represents one unique food
food_matrix = df_joined.pivot_table(
    index='description',
    columns='name',
    values='amount',
    aggfunc='mean'
).fillna(0)

#Vector for the target nutrients 
target_nutrients = [
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg' 
]

filtered_food_matrix = food_matrix[target_nutrients]

#Cosine similarity reccomender function
def recommend_food(patient_vector, food_db, top_n=10):

    #Converting patient vectors to a 2D row array
    vector_array_2d = np.array(patient_vector).reshape(1, -1)

    #Calculating the Cosine Similarity across all matrix rows simultaneously 
    similar_scores = cosine_similarity(vector_array_2d, food_db)[0]

    #Compiling the results into a readable output table
    results_df = food_db.copy()
    results_df['Match Score (%)'] = np.round(similar_scores * 100, 2)

    #Sorting from highest geometric match to lowest
    return results_df.sort_values(by='Match Score (%)', ascending=False).head(top_n)

patient_vector_test = [35.0, 20.0, 400.0]

top_reccomendations = recommend_food(patient_vector_test, filtered_food_matrix, top_n=5)
print("Top matching food reccomended:")
print(top_reccomendations)


Top matching food reccomended:
name                                                Fiber, total dietary  \
description                                                                
Edamame, frozen, prepared                                         6.2326   
Restaurant, Latino, pupusas con frijoles (pupus...                5.8000   
Bread, white, commercially prepared                               2.3000   
Sauce, pasta, spaghetti/marinara, ready-to-serve                  1.8000   
Restaurant, Latino, tamale, pork                                  2.4000   

name                                                Fatty acids, total polyunsaturated  \
description                                                                              
Edamame, frozen, prepared                                                       4.0726   
Restaurant, Latino, pupusas con frijoles (pupus...                              2.8800   
Bread, white, commercially prepared                                         

C:\Users\Lily Jayne Baxendale\AppData\Local\Temp\ipykernel_27768\3457383676.py:5: DtypeWarning: Columns (0: footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  df_food_nutrient = pd.read_csv('food_nutrient.csv')


In [ ]:
#